In [1]:
from scanpy import AnnData
import scanpy as sc
import pandas as pd
import dotenv
import os
import phate
import zipfile
import numpy as np 
from scipy.sparse import csr_matrix
from scipy.sparse import issparse
dotenv.load_dotenv()
DATA_DIR = os.getenv('DATA_DIR')

## load data and unzip

In [2]:
# zip_path = os.path.join(DATA_DIR, "gupta/preprocessed_perturb_seq_filtered_genes_10.h5ad.zip")
adata_path = os.path.join(DATA_DIR, "gupta/preprocessed_perturb_seq_filtered_genes_10.h5ad")
top_gene_path = os.path.join(DATA_DIR, "gupta/subset_top_5000_hvg.h5ad")

In [3]:
# # load the data and unzip
# unzip_path = os.path.join(DATA_DIR, "gupta/")
# with zipfile.ZipFile(zip_path, 'r') as z:
#     z.extractall(unzip_path)

In [4]:
# load the data
adata = sc.read_h5ad(adata_path)
top_genes = sc.read_h5ad(top_gene_path)

## top 5000 hvg batch remove

In [5]:
top_genes

AnnData object with n_obs × n_vars = 214449 × 5000
    obs: 'batch', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'perturbed_gene', 'perturbation_barcode', 'cell_barcode', 'experiment', 'batch_name'
    var: 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'batch_colors', 'hvg', 'log1p', 'neighbors', 'pca', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    layers: 'counts'
    obsp: 'connectivities', 'distances'

In [6]:
# remove the batch effect on them using combat
# sc.pp.combat(adata, key='batch')
sc.pp.combat(top_genes, key='batch')

/gpfs/gibbs/project/gerstein/kx44/conda_envs/new_scanpy_env/lib/python3.9/site-packages/scanpy/preprocessing/_combat.py:352: RuntimeWarning: divide by zero encountered in divide
  (abs(g_new - g_old) / g_old).max(), (abs(d_new - d_old) / d_old).max()


In [7]:
# Extract the data matrix
data = top_genes.X if isinstance(top_genes.X, (np.ndarray, np.matrix)) else top_genes.X.toarray()

In [ ]:
# Run PHATE 2D
phate_op = phate.PHATE(n_components=2)
phate_embedding = phate_op.fit_transform(data)
top_genes.obsm["X_phate_2D"] = phate_embedding

Calculating PHATE...
  Running PHATE on 214449 observations and 5000 variables.
  Calculating graph and diffusion operator...
    Calculating PCA...


    Calculated PCA in 34.92 seconds.
    Calculating KNN search...


In [ ]:
# Run PHATE 3D
phate_op = phate.PHATE(n_components=3)
phate_embedding = phate_op.fit_transform(data)
top_genes.obsm["X_phate_3D"] = phate_embedding

In [ ]:
# save the data
top_genes.write_h5ad(os.path.join(DATA_DIR, "gupta/subset_top_5000_hvg_batch_rm.h5ad"))

## filtered genes batch remove

In [ ]:
adata.obs['batch'] = adata.obs['batch'].astype(str)
sc.pp.combat(adata, key="batch")

In [ ]:
sc.tl.pca(adata, svd_solver="arpack")
sc.pp.neighbors(adata, use_rep="X_pca")
sc.tl.umap(adata)

In [ ]:
# Extract the data matrix
data = adata.X if isinstance(adata.X, (np.ndarray, np.matrix)) else adata.X.toarray()

In [ ]:
# Run PHATE 2D
phate_op = phate.PHATE(n_components=2)
phate_embedding = phate_op.fit_transform(data)
adata.obsm["X_phate_2D"] = phate_embedding

In [ ]:
# Run PHATE 3D
phate_op = phate.PHATE(n_components=3)
phate_embedding = phate_op.fit_transform(data)
adata.obsm["X_phate_3D"] = phate_embedding

In [ ]:
if not isinstance(adata.X, csr_matrix):
    adata.X = csr_matrix(adata.X)

In [ ]:
# Clear unnecessary layers
if 'counts' in adata.layers:
    del adata.layers['counts']

# Clear unnecessary metadata
for key in ['neighbors', 'pca', 'hvg']:
    if key in adata.uns:
        del adata.uns[key]

In [ ]:
# Convert adata.X to float32
if issparse(adata.X):
    adata.X = adata.X.astype('float32')
else:
    adata.X = adata.X.astype(np.float32)

# Convert numeric columns in obs to float32
for col in adata.obs.columns:
    if pd.api.types.is_numeric_dtype(adata.obs[col]):  # Explicitly check for numeric dtype
        adata.obs[col] = adata.obs[col].astype(np.float32)
    elif pd.api.types.is_categorical_dtype(adata.obs[col]):  # Check for categorical dtype
        print(f"Skipping categorical column in obs: {col}")

# Convert numeric columns in var to float32
for col in adata.var.columns:
    if pd.api.types.is_numeric_dtype(adata.var[col]):  # Explicitly check for numeric dtype
        adata.var[col] = adata.var[col].astype(np.float32)
    elif pd.api.types.is_categorical_dtype(adata.var[col]):  # Check for categorical dtype
        print(f"Skipping categorical column in var: {col}")

# Convert all numerical values in uns to float32
for key, value in adata.uns.items():
    if isinstance(value, np.ndarray) and np.issubdtype(value.dtype, np.number):
        adata.uns[key] = value.astype(np.float32)

# Convert obsm values to float32
for key, value in adata.obsm.items():
    if isinstance(value, np.ndarray) and np.issubdtype(value.dtype, np.number):
        adata.obsm[key] = value.astype(np.float32)

# Convert varm values to float32
for key, value in adata.varm.items():
    if isinstance(value, np.ndarray) and np.issubdtype(value.dtype, np.number):
        adata.varm[key] = value.astype(np.float32)

# Convert layers to float32
for layer in adata.layers:
    if issparse(adata.layers[layer]):
        adata.layers[layer] = adata.layers[layer].astype('float32')
    else:
        adata.layers[layer] = adata.layers[layer].astype(np.float32)

# Convert obsp to float32
for key, value in adata.obsp.items():
    if issparse(value):
        adata.obsp[key] = value.astype('float32')
    else:
        adata.obsp[key] = value.astype(np.float32)

print("Conversion to float32 completed.")

In [ ]:
adata.write_h5ad("../datasets/gupta/preprocessed_perturb_seq_filtered_genes_10_batch_rm.h5ad")